# 4 · Os juros vão para banqueiros? E produzem capital produtivo?

**Pergunta dupla:**
(a) o **rendimento** da dívida se concentra no setor financeiro?
(b) algum recurso desses juros vira **capital produtivo** (investimento real)?

**Plano de evidência:**
- (a) participação do setor financeiro no estoque e no fluxo estimado de juros (base do N2);
- (b) comparação **r vs g** (juro real ex ante × crescimento nominal do PIB): quando r > g,
  a dívida cresce **sem esforço fiscal** adicional e o capital migra para títulos
  soberanos em vez de crédito/investimento;
- ações: Tesouro Direto (pessoas físicas) como contraponto à tese "o povo financia o Tesouro".

**Fontes:** RMD Anexo 2.7 (detentores); SGS 4616/4380; Focus (Olinda).

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
dz = ind.detentores_anuais()
part = dz.pivot_table(index="data", columns="detentor", values="participacao") * 100
part.tail(3).round(1)

detentor,Fundos de Investimento,Governo,Instituições Financeiras,Não-residentes,Outros,Previdência,Seguradoras,Tesouro Direto
data,,,,,,,,
2023-12-31,23.50,3.80,29.70,9.50,4.50,23.00,4.00,2.00
2024-12-31,21.70,3.40,29.50,10.20,5.10,23.90,4.00,2.30
2025-12-31,20.80,2.90,32.90,10.30,4.30,22.80,3.50,2.60


In [3]:
medias = []
for faixa, rotulo in [((2007, 2013), "2007-2013"), ((2014, 2019), "2014-2019"),
                      ((2020, 2023), "2020-2023"), ((2024, 2025), "2024-2025")]:
    d = dz[(dz.ano >= faixa[0]) & (dz.ano <= faixa[1])]
    md = d.groupby("detentor")["participacao"].mean() * 100
    md.name = rotulo
    medias.append(md)
comp = pd.concat(medias, axis=1)
print("Participação média no estoque (%) por período:")
print(comp.round(1).to_string())

Participação média no estoque (%) por período:
                          2007-2013  2014-2019  2020-2023  2024-2025
detentor                                                            
Fundos de Investimento        26.10      23.40      24.40      21.20
Governo                        4.70       5.00       4.10       3.10
Instituições Financeiras      34.00      24.60      29.50      31.20
Não-residentes                10.50      14.30       9.70      10.30
Outros                         4.60       4.00       4.30       4.70
Previdência                   16.20      23.10      22.60      23.30
Seguradoras                    3.50       4.30       3.90       3.70
Tesouro Direto                 0.30       1.20       1.70       2.40


In [4]:
fig, ax = plt.subplots(figsize=(11, 5))
financeiro = part[["Instituições Financeiras", "Fundos de Investimento",
                   "Seguradoras"]].sum(axis=1)
financeiro.plot(ax=ax, color=VERMELHO, label="sistema financeiro (IF + Fundos + Seguros)")
part["Previdência"].plot(ax=ax, color=AZUL, label="Previdência")
part["Não-residentes"].plot(ax=ax, color=ROXO, label="Não-residentes")
part["Tesouro Direto"].plot(ax=ax, color=VERDE, label="Tesouro Direto (pessoas físicas)")
ax.set_ylabel("% do estoque")
ax.set_title("Concentração do rendimento: quem tem a dívida federal hoje")
ax.legend(fontsize=9)
salvar(fig, "04_concentracao")

fig -> viz/figs/04_concentracao.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/04_concentracao.png')

In [5]:
# i vs g: custo médio do estoque (nominal) vs crescimento do PIB nominal
cm = ind.custo_medio_dpf().set_index("data")["taxa_pct_aa"]
i_nom = cm.resample("YE").last()
i_nom.index = i_nom.index.year
v = ind.juros_anuais()
v = v[(v.ano >= 2011) & (v.ano < 2026)].set_index("ano")
g_nom = v["pib_mi"].pct_change() * 100
# r vs g em termos reais: g_real ≈ crescimento nominal − IPCA 12m (média do ano)
mm = ind.selic_real_mensal()
r_real = mm["juro_real_ex_ante"].dropna().groupby(lambda d: d.year).mean()
ipca = mm["ipca_12m"].dropna().groupby(lambda d: d.year).mean()
g_real = g_nom - ipca
tab = pd.DataFrame({
    "i (custo médio DPF)": i_nom,
    "g (PIB nominal)": g_nom,
    "i−g": i_nom - g_nom,
    "r (juro real ex ante)": r_real,
    "g (PIB real aprox.)": g_real,
    "r−g": r_real - g_real,
}).loc[2012:2025]
print(tab.round(1).to_string())

      i (custo médio DPF)  g (PIB nominal)   i−g  r (juro real ex ante)  g (PIB real aprox.)    r−g
2012                10.20            10.00  0.20                   3.00                 4.60  -1.60
2013                12.60            10.70  1.90                   2.30                 4.50  -2.20
2014                14.70             8.40  6.30                   4.60                 2.10   2.50
2015                15.70             3.80 12.00                   6.80                -5.30  12.10
2016                 9.80             4.60  5.20                   8.30                -4.20  12.50
2017                10.80             5.00  5.80                   5.80                 1.60   4.20
2018                 7.60             6.40  1.20                   2.50                 2.70  -0.20
2019                 8.20             5.50  2.70                   2.20                 1.80   0.50
2020                 7.20             3.00  4.20                  -0.60                -0.20  -0.30


In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
ax1.bar(tab.index - 0.18, tab.iloc[:, 0], width=0.36, color=VERMELHO,
        label="i (custo médio DPF)")
ax1.bar(tab.index + 0.18, tab.iloc[:, 1], width=0.36, color=AZUL,
        label="g (PIB nominal)")
ax1.axhline(0, color="k", lw=1)
ax1.set_title("Nominal: i vs g")
ax1.set_ylabel("% a.a.")
ax2.bar(tab.index - 0.18, tab.iloc[:, 3], width=0.36, color=VERMELHO,
        label="r (juro real ex ante)")
ax2.bar(tab.index + 0.18, tab.iloc[:, 4], width=0.36, color=AZUL,
        label="g (PIB real aprox.)")
ax2.axhline(0, color="k", lw=1)
ax2.set_title("Real: r vs g")
for ax in (ax1, ax2):
    ax.legend(fontsize=9)
ax2.set_ylim(-2, 20)
fig.suptitle("Custo da dívida acima do crescimento: mecanismo da expansão automática",
             y=1.02, fontweight="bold")
salvar(fig, "04_i_g")

fig -> viz/figs/04_i_g.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/04_i_g.png')

**Achados verificáveis (P4).**

- O **rendimento** é concentrado: o sistema financeiro (bancos + fundos +
  seguradoras) responde por ~50% do estoque e cerca de metade do fluxo
  estimado de juros — "banqueiros" é uma **descrição, não uma metáfora**,
  do grupo destinatário dominante (precisão: "banco" aqui inclui gestão de
  fundos e seguros, não só banco comercial).
- **i > g quase sempre**: exceto o choque inflacionário de 2021–22 (que
  inflou o PIB nominal), o custo médio do estoque da dívida **excede o
  crescimento do PIB nominal em quase todos os anos desde 2012** — em
  2024–25, por ~5–6 p.p. É a condição clássica de **dominância fiscal /
  dívida auto-alimentada**: o estoque cresce pelo juro rolado, sem
  necessidade de emissão primária grande.
- **r > g em termos reais desde 2022**: com PIB real crescendo ~2,5–3% e
  juro real ex ante de 7–9%, o retorno do papel público supera o
  crescimento — racionalmente, capital privado migra para o título
  soberano; crédito e investimento perdem o embate de arbitragem.
- Sobre **capital produtivo**: nos dados públicos de fluxo, não existe
  caminho em que o dispêndio com juros se converta em investimento real —
  é rolagem (N3) + transferência a detentores (N2). Não há evidência
  **a favor** de que juros altos "financiam" capital produtivo (público
  ou privado); há dados consistentes com o efeito inverso — o soberano
  "rouba" a poupança do crédito privado quando rende 9% real sem risco.